In [ ]:
import deepinv as dinv
import torch
import matplotlib.pyplot as plt
from deepinv.utils.demo import load_dataset
from torchvision import transforms
import numpy as np
from sapg import SAPG
from priors import WaveletPrior, GSDPrior, L2, CombinedPrior, L1Prior, CRRPrior, WeightedTikhonovPrior
from prior_comparison import DegradedLikelihood
import tqdm
from utils import device
from models.utils import load_model
from scipy.stats import ortho_group

In [ ]:
def create_covariance_matrix(d, v1, v2, sig, reg_strength, ratio=0.25):
    # create a random orthogonal matrix
    d = int(d)
    Q = ortho_group.rvs(d)
    D1  = np.zeros(d)
    ns1 = int(ratio*d)
    D1[:ns1] = v1
    D1[ns1:] = v2
    
    D, D_inv = np.diag(D1), np.diag(1/D1)
    N_inv = np.diag(1/(np.ones(d)/sig**2 + reg_strength/D1)) 
    # create the covariance matrix
    return Q, D, Q @ D_inv @ Q.T, Q @ N_inv @ Q.T, ns1, d-ns1

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

v1 = 1.  #  strongest variance
v2 = 0.01  # weakest variance
noise_level = 0.05
reg_strength = 1.

def generate_measurements(d):
    d = int(d)
    Q, D, cov_inv, N_inv, d1, d2 = create_covariance_matrix(d, v1, v2, noise_level, reg_strength)
    x = torch.tensor((Q @ np.sqrt(D) @ np.random.normal(size=d)).astype(np.float32)).to(device).reshape((1, 1, d))
    
    p = dinv.physics.LinearPhysics(  # identity
        img_size=(1, d),
        device=device, 
        noise_model=dinv.physics.GaussianNoise(sigma=noise_level))
    
    y = p(x) 
    return (y, x, p, d1, d2,
            torch.tensor(cov_inv.astype(np.float32), device=device),
            torch.tensor(N_inv.astype(np.float32), device=device))

def compute_evidence(d1, d2, v1, v2, noise_level, reg_strength, y, cov_inv, N_inv, mlog=True):
    d = d1 + d2
    # +log of evidence
    res = -torch.norm(y)**2/2/noise_level**2
    res += 0.5*torch.sum(torch.reshape(y, (-1, 1))*(N_inv@torch.reshape(y, (-1, 1)))) / noise_level**4
    res -= d1*0.5*np.log(1/noise_level**2 + reg_strength/v1)  # first part of 1/sqrt(det N)
    res -= d2*0.5*np.log(1/noise_level**2 + reg_strength/v2)  # second part of sqrt(det N)
    res -= d1*0.5*np.log(v1) + d2*0.5*np.log(v2) # volume of the prior first part : det M
    res += d*0.5*np.log(reg_strength)  # volume of the prior 2nd part
    res += d*np.log(noise_level) - 0.5*d*np.log(2*np.pi)  # normalization of likelihood term
    if mlog == False:
        res = torch.exp(res)
    else:
        res = - res
    return res
                
# Lipschitz constant of nabla f
L_f = 1 / noise_level**2 / 2.  # divide by 2 because std is sqrt(2)sigma

In [ ]:
# Lipschitz constant of grad g, which is reg_str * norm(cov-1) = reg_str * norm(cov-1)
L_g = reg_strength * max(1/v1, 1/v2)
L =  L_f + L_g

# Stepsize of MCMC algorithm
gamma = 0.98*1/L
print("gamma: {}".format(gamma))

### Define the priors and associated constants

In [ ]:
nval = 10
dims = torch.logspace(2, 3., nval, dtype=torch.int)
hist, evidences = np.zeros(nval), np.zeros(nval)
nb_steps, burnin_ratio = 500000, 0.9
post_hist_traces = []

for i in range(nval):
    d = dims[i]
    y, x, p, d1, d2, cov_inv, N_inv = generate_measurements(d)
    g = WeightedTikhonovPrior(torch.tensor(reg_strength), cov_inv)
    L_g = d * max(1/v1, 1/v2)
    gamma = 0.98*1/(L_f + L_g)
    dl = DegradedLikelihood(y, g, p, noise_level, gamma, X_init=p.A_A_adjoint(y).to(device).clone(),
                            lam_reg=None, project=None)
    
    print("d={}".format(d))
    
    X_post_trace, post_hist, lik_trace, lik_mean = dl.compute_test2(nb_steps, burnin_ratio=burnin_ratio, log_stats=True, thinning=1000, tol=1e-2, patience=10)
    evidences[i] = compute_evidence(d1, d2, v1, v2, noise_level, reg_strength, y, cov_inv, N_inv, mlog=True)
    hist[i] = lik_mean + 0.5*d*np.log(2*np.pi) + d*np.log(np.sqrt(2)*noise_level)  # lik_mean is E_eps(p(y+/y-)) without normalization constant
    post_hist_traces.append(post_hist.numpy())
    print("likelihood: ", hist[i], "  evidence: ", evidences[i])

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(25,5))

ax[0].plot(dims, hist, label=r'$\mathbb{E}_\varepsilon(p(y^+/\;y^-))$')
ax[0].plot(dims, evidences, '--', label=r'$p(y)$')

ax[0].set_xlabel(r'$d$')
c1, c2 = np.array((0.1, 0.2, 0.5)), np.array((0.9, 0.3, 0.55))
for i in range(len(post_hist_traces)):
    if i == 0 or i == nval - 1:
        label = r'$d = {:.3f}'.format(dims[i])
    else:
        label = None
    t = i / nval
    ax[1].plot(post_hist_traces[i], color=(t*c1 + (1-t)*c2), linestyle='-.',label=label)

ax[0].legend(), ax[1].legend()
ax[0].set_yscale('log')

plt.savefig('figs/test2_evidence3.pdf')